# Classical baseline: logistic regression (Tâche 4)
The director's question: is a neural network worth it, or would a simpler classical model do the job?
Before comparing, we need a classical model trained on **the same paintings, with the same split** as `mlp.ipynb` and `resnet.ipynb`.

The model is always a **logistic regression** (scikit-learn): one linear layer + softmax, trained by scikit-learn, nothing learned in PyTorch. We only change what it receives as input:

1. **Pixels**: the 64 x 64 images of the MLP, flattened. Same input as the MLP, so we see what the hidden layers of the MLP bring.
2. **Colour histogram**: how much of each colour the painting contains. No neural network at all, the cheapest option for a kiosk.
3. **Frozen ResNet features**: the 512 numbers that ResNet-18 (trained on ImageNet) computes for each painting. ResNet is only used to describe the image, it is never trained on our paintings.

For each one: the regularisation strength `C` is chosen on the validation set, then test accuracy and macro-F1 on the 250 test paintings.

**On Colab only** (GPU not needed but faster for the ResNet features: *Runtime → Change runtime type → T4 GPU*): uncomment and run the cell below first. Same cell as in `resnet.ipynb`: change `DATA_ON_DRIVE` to where the `wikiart_style5` folder is on your Drive. Locally, skip it.

In [ ]:
#  --- Colab only ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_ON_DRIVE = "/content/drive/MyDrive/wikiart_style5"     # <- adapt
# !git clone -q https://github.com/raniamehria/museum-guide.git /content/museum-guide
# %cd /content/museum-guide
# !git checkout -q rania                                                                       # branch with baseline.py
# !rm -rf data/wikiart_style5 && mkdir -p data && cp -r "{DATA_ON_DRIVE}" data/wikiart_style5
# !find data/wikiart_style5 -name "*.jpg" | wc -l                                             # must print 1500

## Setup
Same seeds and same class order as the other notebooks.

In [ ]:
import os
import sys
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")                  # run from the repo root, where data/ and splits/ are
sys.path.insert(0, "src")           # find the museum_guide package without installing it

import json
import time

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix

from museum_guide.data import load_images
from museum_guide.baseline import (C_GRID, pixel_features, colour_histogram, load_frozen_resnet,
                                   resnet_features, fit_logreg, scores, count_logreg_parameters)
from museum_guide.benchmark import sklearn_size_mb

np.random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

CLASSES = ["Baroque", "Cubism", "Impressionism", "Pop_Art", "Ukiyo_e"]
N_TEST = 250
# results of the other notebooks, to compare on the same 250 test paintings
MLP_ACC = 0.356             # tuned MLP, mlp.ipynb (Luli, PR #4)
RESNET_HEAD_ACC = 0.732     # ResNet-18 head only (step A), resnet.ipynb (Wei-Ling, PR #3)
print("device:", device)

## Data
We use the shared split `splits/split.csv` (1 000 train / 250 validation / 250 test, grouped by artist), so the test paintings are exactly the ones of the MLP and the ResNet.
We do not use `get_dataloaders` here: scikit-learn wants all the images at once in a numpy array, not mini-batches.

In [ ]:
X, y, meta = load_images()                       # (1500, 3, 128, 128) in [0, 1], same order as metadata.csv
split = pd.read_csv("splits/split.csv")
assert (split["path"] == meta["path"]).all(), "split.csv does not match metadata.csv"

masks = {s: torch.tensor((split["split"] == s).values) for s in ["train", "val", "test"]}
y_train, y_val, y_test = (y[masks[s]].numpy() for s in ["train", "val", "test"])
print({s: int(m.sum()) for s, m in masks.items()})
assert len(y_test) == N_TEST
print(pd.crosstab(meta["label"], split["split"]))

## Helper functions
`run_baseline` does the same thing for the 3 inputs: choose C on validation, then measure on test. `plot_confusion` is the same plot as in `resnet.ipynb`.

In [ ]:
results = {}       # one entry per baseline, saved at the end


def run_baseline(name, F_train, F_val, F_test):
    print(f"{name}: {F_train.shape[1]:,} features per painting")
    t0 = time.time()
    model, C, table = fit_logreg(F_train, y_train, F_val, y_val)
    s = scores(model, F_test, y_test)
    results[name] = {"C": C, "n_features": int(F_train.shape[1]), "parameters": count_logreg_parameters(model),
                     "val_acc": max(r["val_acc"] for r in table), "test_acc": s["accuracy"],
                     "test_macro_f1": s["macro_f1"], "fit_seconds": time.time() - t0, "c_table": table}
    print(f"test accuracy {s['accuracy']:.3f} | macro-F1 {s['macro_f1']:.3f} | chance 0.20")
    return model, s["pred"]


def plot_confusion(pred, title):
    cm = confusion_matrix(y_test, pred, labels=range(len(CLASSES)))
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xticks(range(len(CLASSES)), CLASSES, rotation=90); plt.yticks(range(len(CLASSES)), CLASSES)
    plt.xlabel("predicted"); plt.ylabel("true"); plt.title(title); plt.colorbar(); plt.show()
    off = cm.copy()
    np.fill_diagonal(off, 0)
    t, p = np.unravel_index(off.argmax(), off.shape)
    for i, c in enumerate(CLASSES):
        print(f"{c:14s} accuracy {cm[i, i] / cm[i].sum():.2f}")
    if off.max() > 0:
        print(f"most frequent mistake: {CLASSES[t]} predicted as {CLASSES[p]} ({cm[t, p]} times)")

## Baseline 1: logistic regression on the pixels
Same 64 x 64 images as the MLP. The only difference with the MLP is that there are no hidden layers: the model is linear.
If this is close to the MLP (0.356), the hidden layers of the MLP do not bring much on our paintings.

In [ ]:
P_train, P_val, P_test = (pixel_features(X[masks[s]]) for s in ["train", "val", "test"])
logreg_pixels, pred_pixels = run_baseline("logreg_pixels", P_train, P_val, P_test)
print(f"vs MLP on the same input: {(results['logreg_pixels']['test_acc'] - MLP_ACC) * 100:+.1f} points")

## Baseline 2: logistic regression on a colour histogram
Each channel (R, G, B) is cut into 8 levels, which gives 8 x 8 x 8 = 512 colour boxes. For each painting we count the share of its pixels in each box.
No neural network at all: computing a histogram is very cheap, which is interesting for a modest kiosk CPU.

Before training, a quick look at what this input can "see": the average colour of each style in the training set.

In [ ]:
mean_colour = [X[masks["train"]][y[masks["train"]] == k].mean(dim=(0, 2, 3)) for k in range(len(CLASSES))]
plt.figure(figsize=(7, 1.6))
for k, c in enumerate(mean_colour):
    plt.fill_between([k, k + 0.9], 0, 1, color=c.numpy())
    plt.text(k + 0.45, -0.25, CLASSES[k], ha="center", fontsize=9)
plt.axis("off"); plt.title("average colour of each style (train set)"); plt.show()

In [ ]:
H_train, H_val, H_test = (colour_histogram(X[masks[s]]) for s in ["train", "val", "test"])
assert np.allclose(H_train.sum(axis=1), 1.0)       # each histogram must sum to 1
logreg_colours, pred_colours = run_baseline("logreg_colours", H_train, H_val, H_test)

## Baseline 3: logistic regression on frozen ResNet-18 features
ResNet-18 pretrained on ImageNet, without its last layer: each painting becomes 512 numbers. ResNet is **frozen** (never trained here), only the logistic regression learns.

This is the same model as step A of `resnet.ipynb` (a linear layer on the frozen backbone), trained by scikit-learn instead of Adam. So the test accuracy should be close to Wei-Ling's 0.732: it is a check that our two notebooks agree.

In [ ]:
backbone = load_frozen_resnet()
t0 = time.time()
R_train, R_val, R_test = (resnet_features(X[masks[s]], backbone, device=device) for s in ["train", "val", "test"])
print(f"features computed in {time.time() - t0:.0f} s, shape {R_train.shape}")
logreg_resnet, pred_resnet = run_baseline("logreg_resnet", R_train, R_val, R_test)
print(f"vs ResNet step A (resnet.ipynb): {(results['logreg_resnet']['test_acc'] - RESNET_HEAD_ACC) * 100:+.1f} points")

## Choosing C: regularisation
In scikit-learn, `C` is the inverse of the L2 penalty of session 6 (C = 1 / λ): a small C keeps the weights small, a large C lets the model fit the training set.
If the train accuracy goes to 1.0 while the validation accuracy drops, the model overfits, as in the course.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for ax, name in zip(axes, results):
    table = pd.DataFrame(results[name]["c_table"])
    ax.plot(table["C"], table["train_acc"], "o-", label="train")
    ax.plot(table["C"], table["val_acc"], "o-", label="validation")
    ax.axvline(results[name]["C"], color="grey", linestyle="--")
    ax.set_xscale("log"); ax.set_xlabel("C (log scale)"); ax.set_title(name); ax.legend()
axes[0].set_ylabel("accuracy")
fig.suptitle("Train vs validation accuracy for each C (dashed line = kept C)")
plt.tight_layout(); plt.show()

## Errors of the best baseline

In [ ]:
best = max(results, key=lambda n: results[n]["test_acc"])
best_pred = {"logreg_pixels": pred_pixels, "logreg_colours": pred_colours, "logreg_resnet": pred_resnet}[best]
plot_confusion(best_pred, f"{best} (test set)")

## Results
With 250 test paintings, one standard deviation of the accuracy is about 3 points (`sqrt(p(1 - p) / 250)`), so differences smaller than that are noise.

In [ ]:
table = pd.DataFrame([
    ["chance (1 / 5)", 0.20, None, 0],
    ["MLP, 3 ReLU layers (mlp.ipynb)", MLP_ACC, None, None],
    ["ResNet-18 head only (resnet.ipynb)", RESNET_HEAD_ACC, 0.734, 2565],
] + [[n, r["test_acc"], r["test_macro_f1"], r["parameters"]] for n, r in results.items()],
    columns=["model", "test accuracy", "macro-F1", "trained parameters"])
table["± 1 std"] = np.sqrt(table["test accuracy"] * (1 - table["test accuracy"]) / N_TEST)
display(table.style.format({"test accuracy": "{:.3f}", "macro-F1": "{:.3f}", "± 1 std": "{:.3f}",
                            "trained parameters": "{:,.0f}"}, na_rep="-").hide(axis="index"))

Save the numbers for `comparison.ipynb` (the json is small and goes in Git) and the 3 models for the CPU measures (they stay out of Git, `models/` is ignored like in `resnet.ipynb`).

In [ ]:
os.makedirs("reports", exist_ok=True)
for name, model in [("logreg_pixels", logreg_pixels), ("logreg_colours", logreg_colours), ("logreg_resnet", logreg_resnet)]:
    results[name]["size_mb"] = sklearn_size_mb(model, f"models/{name}.pkl")
with open("reports/baseline_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print({n: f"{r['size_mb']:.2f} MB" for n, r in results.items()})

## Conclusion
The cell below prints the measured numbers (to copy into the slides).

In [ ]:
r = results
print(f'''- Pixels (same input as the MLP, {r["logreg_pixels"]["n_features"]:,} features): test accuracy {r["logreg_pixels"]["test_acc"]:.3f}, macro-F1 {r["logreg_pixels"]["test_macro_f1"]:.3f}
  ({(r["logreg_pixels"]["test_acc"] - MLP_ACC) * 100:+.1f} points vs the MLP {MLP_ACC}).
- Colour histogram ({r["logreg_colours"]["n_features"]} features, no network): test accuracy {r["logreg_colours"]["test_acc"]:.3f}, macro-F1 {r["logreg_colours"]["test_macro_f1"]:.3f}.
- Frozen ResNet features (512 features): test accuracy {r["logreg_resnet"]["test_acc"]:.3f}, macro-F1 {r["logreg_resnet"]["test_macro_f1"]:.3f}
  ({(r["logreg_resnet"]["test_acc"] - RESNET_HEAD_ACC) * 100:+.1f} points vs ResNet step A {RESNET_HEAD_ACC}, same model trained with Adam).
- Kept C: pixels {r["logreg_pixels"]["C"]}, colours {r["logreg_colours"]["C"]}, ResNet {r["logreg_resnet"]["C"]}.
- One standard deviation with 250 test paintings: about 3 points.''')